# 데이터 수집 랩

제품 데이터가 포함된 CSV 파일을 읽어옵니다.

##### 작업
1. 스키마 추론을 사용하여 읽기
2. 사용자 정의 스키마를 사용하여 읽기
3. 스키마를 DDL 형식 문자열로 사용하여 읽기
4. 델타 형식을 사용하여 쓰기

In [0]:
%run ./Includes/Classroom-Setup-00.08L

Resetting the learning environment:
| dropping the schema "3dt005_nuk5_da_delp"...(0 seconds)
| removing the working directory "dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineer-learning-path"...(0 seconds)

Skipping install of existing datasets to "dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v02"

Validating the locally installed datasets:
| listing local files...(3 seconds)
| validation completed...(3 seconds total)

Creating & using the schema "3dt005_nuk5_da_delp" in the catalog "hive_metastore"...(0 seconds)

Predefined tables in "3dt005_nuk5_da_delp":
| -none-

Predefined paths variables:
| DA.paths.working_dir:  dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineer-learning-path
| DA.paths.user_db:      dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineer-learning-path/database.db
| DA.paths.datasets:     dbfs:/mnt/dbacademy-datasets/data-engineer-learning-path/v02
| DA.paths.products_csv: dbfs:/mnt/dbacademy-users/3dt005@msacad


### 1. 스키마 추론을 사용한 읽기
- 변수 **`single_product_csv_file_path`**에 지정된 파일 경로를 사용하여 DBUtils 메서드 **`fs.head`**로 첫 번째 CSV 파일을 확인합니다.
- 변수 **`products_csv_path`**에 지정된 파일 경로에 있는 CSV 파일에서 읽어 **`products_df`**를 생성합니다.

- 첫 번째 줄을 헤더로 사용하고 스키마를 추론하도록 옵션을 구성합니다.

In [0]:
# TODO

single_product_csv_file_path = f"{DA.paths.products_csv}/part-00000-tid-1663954264736839188-daf30e86-5967-4173-b9ae-d1481d3506db-2367-1-c000.csv"
print(dbutils.fs.head(single_product_csv_file_path))

products_csv_path = DA.paths.products_csv
products_df = (spark
               .read
               .option("header", True)
               .option("inferSchema", True)
               .csv(products_csv_path)
              )

products_df.printSchema()

item_id,name,price
M_PREM_Q,Premium Queen Mattress,1795.0
M_STAN_F,Standard Full Mattress,945.0
M_PREM_F,Premium Full Mattress,1695.0

root
 |-- item_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- price: double (nullable = true)




**1.1: CHECK YOUR WORK**

In [0]:
assert(products_df.count() == 12)
print("All test pass")

All test pass



### 2. 사용자 정의 스키마로 읽기
열 이름과 데이터 형식을 지정하여 **`StructType`**을 생성함으로써 스키마를 정의합니다.

In [0]:
# TODO
from pyspark.sql.types import DoubleType, StringType, StructType, StructField

user_defined_schema = StructType([
    StructField("item_id", StringType(), True),
    StructField("name", StringType(), True),
    StructField("price", DoubleType(), True)
])

products_df2 = (spark
                .read
                .option("header", True)
                .schema(user_defined_schema)
                .csv(products_csv_path)
               )


**2.1: CHECK YOUR WORK**

In [0]:
assert(user_defined_schema.fieldNames() == ["item_id", "name", "price"])
print("All test pass")

All test pass


In [0]:
from pyspark.sql import Row

expected1 = Row(item_id="M_STAN_Q", name="Standard Queen Mattress", price=1045.0)
result1 = products_df2.first()

assert(expected1 == result1)
print("All test pass")

All test pass



### 3. DDL 형식 문자열로 읽기

In [0]:
# TODO
ddl_schema = "item_id string, name string, price double"

products_df3 = (spark
                .read
                .option("header", True)
                .schema(ddl_schema)
                .csv(products_csv_path)
               )


**3.1: CHECK YOUR WORK**

In [0]:
assert(products_df3.count() == 12)
print("All test pass")

All test pass



### 4. Delta에 쓰기
변수 **`products_output_path`**에 지정된 파일 경로에 **`products_df`**를 씁니다.

In [0]:
# TODO
products_output_path = DA.paths.working_dir + "/delta/products"
(products_df
 .write
 .format("delta")
 .mode("overwrite")
 .save(products_output_path)
)


**4.1: CHECK YOUR WORK**

In [0]:
verify_files = dbutils.fs.ls(products_output_path)
verify_delta_format = False
verify_num_data_files = 0
for f in verify_files:
    if f.name == "_delta_log/":
        verify_delta_format = True
    elif f.name.endswith(".parquet"):
        verify_num_data_files += 1

assert verify_delta_format, "Data not written in Delta format"
assert verify_num_data_files > 0, "No data written"
del verify_files, verify_delta_format, verify_num_data_files
print("All test pass")

All test pass



### Clean up classroom

In [0]:
DA.cleanup()

Resetting the learning environment:
| dropping the schema "3dt005_nuk5_da_delp"...(1 seconds)
| removing the working directory "dbfs:/mnt/dbacademy-users/3dt005@msacademy.msai.kr/data-engineer-learning-path"...(0 seconds)

Validating the locally installed datasets:
| listing local files...(3 seconds)
| validation completed...(3 seconds total)
